# MERIT — Layer 4: utility-gated lifecycle

Layer 4 decides about the **store**, not about a query:

- **Promote** — which entries sit in the always-injected core
- **Demote** — which core entries lose their seat
- **Forget** — which entries are evicted when the store must shrink

Global utility is the right signal here, even though Layer 3 showed it is the
wrong one for per-query retrieval. "Should this entry be in the core?" has no
query. Layer 3 failed because it used a query-free number for a per-query
decision; Layer 4 uses it for the decision it actually answers.

**The core score** is expected gain per turn, per token:

```
expected_gain = utility x eligibility_rate      (how often it's relevant)
```

The incumbent rule ranks on `eligibility_rate` alone. Same retrievability data —
MERIT multiplies in measured effect.

Sections 1–5 are mock, zero tokens. Section 6 is the live run.

In [1]:
import sys, pathlib, contextlib, io
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()))
import pandas as pd
pd.set_option("display.width", 180)
from merit import Config, Layer1Runner, build_store, build_tasks
from merit.clients import MockClient
from merit import lifecycle as LC, evaluate as E, credit as C

store, tasks = build_store(), build_tasks()
planted = {e: v.planted for e, v in store.entries.items()}
family = pd.DataFrame([{"task_id": t.task_id, "family": t.family} for t in tasks])
misranked = E.misranked_tasks(store, tasks)
SEED = 20261006
def quiet(f):
    with contextlib.redirect_stdout(io.StringIO()):
        return f()
def by_family(t):
    return (t.dropna(subset=["outcome_score"]).merge(family)
             .groupby(["arm","family"])["outcome_score"].mean().unstack().round(3))

## 1. Learning phase

The decisions need a ledger to learn from. This is the ordinary Layer 1 setting:
relevance retrieval, 15% masking, 400 turns.

In [2]:
learn = Config(run_id="l4_learn", backend="mock", mask_p=0.15, max_turns=400,
               seed=SEED, db_path="data/layer4.db")
def go():
    with Layer1Runner(learn, client=MockClient(0.9, SEED)) as r:
        r.run(verbose=False)
        return r.ledger.dataframe("l4_learn")
ledger = quiet(go)
print(len(ledger), "rows,", ledger["turn_id"].nunique(), "turns")

1600 rows, 400 turns


## 2. Both rules, side by side

In [3]:
scores = quiet(lambda: LC.core_scores(ledger, store))
scores["planted"] = scores["entry_id"].map(planted)
scores.sort_values("gain_per_token", ascending=False)

,entry_id,tokens,retrievals,eligibility_rate,utility,ci_lo,ci_hi,measured,n_withheld,promotable,expected_gain,gain_per_token,planted
1,e_build_flag,20,336,0.840,0.386,0.241,0.531,True,47,True,0.3245,0.01622,useful
0,e_test_flag,21,335,0.838,0.389,0.262,0.515,True,58,True,0.3255,0.01550,useful
2,e_deploy_flag,24,112,0.280,0.902,0.799,1.005,True,18,True,0.2526,0.01053,useful
3,e_test_noise,20,286,0.715,0.085,-0.040,0.209,True,39,False,0.0605,0.00302,distractor
4,e_build_noise,20,270,0.675,0.042,-0.066,0.149,True,42,False,0.0281,0.00140,distractor
6,e_misc_1,12,49,0.122,0.090,-0.223,0.402,True,11,False,0.0110,0.00092,irrelevant
7,e_misc_2,13,17,0.043,0.000,NaN,NaN,False,0,False,0.0000,0.00000,irrelevant
8,e_misc_3,13,17,0.043,0.000,NaN,NaN,False,0,False,0.0000,0.00000,irrelevant
9,e_misc_4,12,16,0.040,0.000,NaN,NaN,False,0,False,0.0000,0.00000,irrelevant
5,e_deploy_noise,20,162,0.405,-0.046,-0.186,0.093,True,29,False,-0.0187,-0.00094,distractor


In [4]:
core_recall  = LC.promote(scores, "recall", 3)
core_utility = LC.promote(scores, "utility", 3)
print("core by recall frequency:", core_recall,  [planted[e] for e in core_recall])
print("core by measured utility:", core_utility, [planted[e] for e in core_utility])

core by recall frequency: ['e_build_flag', 'e_test_flag', 'e_test_noise'] ['useful', 'useful', 'distractor']
core by measured utility: ['e_build_flag', 'e_test_flag', 'e_deploy_flag'] ['useful', 'useful', 'useful']


## 3. E4a — Promotion: deploy each core

Each arm injects 3 core entries plus 1 relevance hit from the rest — the same
token budget. Masking off: this measures the policy, not the experiment.

The comparison that matters is **recall core vs utility core**. A core exists in
OpenClaw-style systems either way; the question is how its seats are filled.

In [5]:
def deploy(name, policy_factory=None, st=None, n=240):
    c = Config(run_id=f"l4_{name}", backend="mock", mask_p=0.0, top_k=4,
               max_turns=n, seed=SEED, db_path="data/layer4.db")
    return quiet(lambda: E.run_arm(name, c, policy_factory, MockClient(0.9, SEED), store=st))[0]

e4a = pd.concat([
    deploy("relevance_k4"),
    deploy("core_recall",  lambda s: LC.CorePolicy(s, core_recall,  k_episodic=1)),
    deploy("core_utility", lambda s: LC.CorePolicy(s, core_utility, k_episodic=1)),
])
E.summarize(e4a, misranked)

,arm,turns,outcome,outcome_misranked,outcome_rest,entries_injected,memory_tokens,outcome_per_100_mem_tokens
0,relevance_k4,240,0.838,0.714,0.888,4.0,88.0,0.951
1,core_recall,240,0.633,0.271,0.782,4.0,89.5,0.708
2,core_utility,240,0.800,0.843,0.782,4.0,92.7,0.863


In [6]:
by_family(e4a)

family,build,deploy,test
arm,,,
core_recall,0.938,0.238,0.725
core_utility,0.938,0.738,0.725
relevance_k4,0.938,0.725,0.850


The whole gap is on deploy tasks — exactly where the recall core gave its third
seat to a distractor instead of the deploy flag.

## 4. E4b — Forgetting: shrink the store to 4 entries

In [7]:
keep_recall  = LC.forget(scores, "recall", 4)
keep_utility = LC.forget(scores, "utility", 4)
print("kept by recall frequency:", [planted[e] for e in keep_recall])
print("kept by measured utility:", [planted[e] for e in keep_utility])

e4b = pd.concat([
    deploy("forget_recall",  st=LC.subset_store(store, keep_recall)),
    deploy("forget_utility", st=LC.subset_store(store, keep_utility)),
])
E.summarize(e4b, misranked)

kept by recall frequency: ['useful', 'useful', 'distractor', 'distractor']
kept by measured utility: ['useful', 'useful', 'useful', 'distractor']


,arm,turns,outcome,outcome_misranked,outcome_rest,entries_injected,memory_tokens,outcome_per_100_mem_tokens
0,forget_recall,240,0.596,0.271,0.729,4.0,89.0,0.669
1,forget_utility,240,0.875,0.843,0.888,4.0,93.0,0.941


In [8]:
by_family(e4b)

family,build,deploy,test
arm,,,
forget_recall,0.938,0.000,0.85
forget_utility,0.938,0.838,0.85


Frequency-based eviction forgets the deploy flag — the most valuable entry in
the store — because it is the least often retrieved. Deploy tasks then fail
completely, permanently. This is the failure mode of any LRU- or frequency-based
memory budget.

## 5. E4c — Online lifecycle: can a wrong core correct itself?

Start from the **recall** core (one distractor seated), keep 15% masking on, and
refit every 50 turns: demote entries that fail the gate, promote ones that pass.

Demotion is **power-aware**: an entry is only judged once it has enough
*withheld* trials. Without that guard, every entry looks unproven early on and
useful entries are evicted along with dead weight.

In [9]:
def lifecycle_run(min_withheld, seed):
    pol = LC.LifecyclePolicy(store, core_recall, size=3, k_episodic=1,
                             refit_every=50, demote_min_withheld=min_withheld)
    cfg = Config(run_id=f"l4_life_{min_withheld}_{seed}", backend="mock", mask_p=0.15,
                 top_k=4, max_turns=600, seed=seed, db_path="data/layer4.db")
    def go():
        with Layer1Runner(cfg, store=store, tasks=tasks, client=MockClient(0.9, seed), policy=pol) as r:
            r.run(verbose=False)
            return r.ledger.turns_dataframe(cfg.run_id)
    t = quiet(go).reset_index(drop=True)
    return pol, t

pol, life = lifecycle_run(15, SEED)
for h in pol.history:
    print(f"turn {h['turn']:>3}  core={[planted[e] + ':' + e.split('_')[1] for e in h['core']]}"
          + (f"  demoted={h['demoted']}" if h['demoted'] else "")
          + (f"  promoted={h['promoted']}" if h['promoted'] else ""))

turn   0  core=['useful:build', 'useful:test', 'distractor:test']
turn  50  core=['useful:build', 'useful:test', 'distractor:test']
turn 100  core=['useful:build', 'useful:test']  demoted=['e_test_noise']
turn 150  core=['useful:test']  demoted=['e_build_flag']
turn 200  core=['useful:test', 'distractor:build']  promoted=['e_build_noise']
turn 250  core=['useful:test', 'distractor:build', 'useful:deploy']  promoted=['e_deploy_flag']
turn 300  core=['useful:test', 'useful:deploy', 'useful:build']  demoted=['e_build_noise']  promoted=['e_build_flag']
turn 350  core=['useful:test', 'useful:deploy', 'useful:build']
turn 400  core=['useful:test', 'useful:deploy', 'useful:build']
turn 450  core=['useful:test', 'useful:deploy', 'useful:build']
turn 500  core=['useful:test', 'useful:deploy', 'useful:build']
turn 550  core=['useful:test', 'useful:deploy', 'useful:build']
turn 600  core=['useful:test', 'useful:deploy', 'useful:build']


In [10]:
life["block"] = life.index // 100
life.dropna(subset=["outcome_score"]).groupby("block")["outcome_score"].mean().round(3).to_frame("outcome").T

block,0,1,2,3,4,5
outcome,0.53,0.56,0.6,0.69,0.77,0.73


### Demotion rule: naive vs power-aware, three seeds

`false_demotions` counts useful entries wrongly evicted. It is reduced by the
power-aware rule, not eliminated — an honest limitation.

In [11]:
rows = []
for mw in (0, 15):
    for s in (SEED, SEED + 1, SEED + 2):
        p, t = lifecycle_run(mw, s)
        rows.append({
            "min_withheld": mw, "seed": s,
            "false_demotions": sum(1 for h in p.history for e in h["demoted"] if planted[e] == "useful"),
            "distractor_out_at": next((h["turn"] for h in p.history if "e_test_noise" not in h["core"]), None),
            "deploy_in_at": next((h["turn"] for h in p.history if "e_deploy_flag" in h["core"]), None),
            "final_core_all_useful": [planted[e] for e in p.core].count("useful") == 3,
            "outcome_last_200": round(t.iloc[-200:]["outcome_score"].mean(), 3),
        })
pd.DataFrame(rows)

,min_withheld,seed,false_demotions,distractor_out_at,deploy_in_at,final_core_all_useful,outcome_last_200
0,0,20261006,1,50,550,True,0.630
1,0,20261007,2,50,500,True,0.635
2,0,20261008,2,100,250,True,0.705
3,15,20261006,1,100,250,True,0.750
4,15,20261007,1,100,250,True,0.715
5,15,20261008,1,150,300,True,0.685


## 6. Live run — spends tokens

The learning phase is free: it reuses your `live_400` ledger. Only the
deployment arms call Groq.

**Budget:** 4 arms x 96 turns x ~450 tokens ≈ **175k tokens** — about the same
as the live_400 run. Lower `N_LIVE` if you want it cheaper; below ~60 the arms
get too noisy to compare.

In [ ]:
from dotenv import load_dotenv
import sqlite3
load_dotenv("../.env")

def find_db(run_id):
    """Your live_400 may be in notebooks/data or data/ depending on which runner wrote it."""
    for cand in ("data/merit.db", "../data/merit.db"):
        p = pathlib.Path(cand)
        if not p.exists():
            continue
        con = sqlite3.connect(p)
        try:
            runs = {r[0] for r in con.execute("SELECT run_id FROM runs")}
        except Exception:
            runs = set()
        con.close()
        if run_id in runs:
            return str(p.resolve())
    raise FileNotFoundError(f"no database containing {run_id!r}")

LIVE_DB = find_db("live_400")
print("learning from:", LIVE_DB)
live_ledger = C.load_ledger(LIVE_DB, "live_400")
live_scores = quiet(lambda: LC.core_scores(live_ledger, store))
live_scores["planted"] = live_scores["entry_id"].map(planted)
lc_recall  = LC.promote(live_scores, "recall", 3)
lc_utility = LC.promote(live_scores, "utility", 3)
lk_recall  = LC.forget(live_scores, "recall", 4)
lk_utility = LC.forget(live_scores, "utility", 4)
print("core recall :", [planted[e] for e in lc_recall])
print("core utility:", [planted[e] for e in lc_utility])
print("keep recall :", [planted[e] for e in lk_recall])
print("keep utility:", [planted[e] for e in lk_utility])

In [ ]:
N_LIVE = 96

def live_arm(name, policy_factory=None, st=None):
    c = Config(run_id=f"l4live_{name}", backend="groq", model="openai/gpt-oss-20b",
               mask_p=0.0, top_k=4, max_turns=N_LIVE, max_total_tokens=60_000,
               db_path="data/merit.db")
    return E.run_arm(name, c, policy_factory, None, store=st, verbose=True)[0]

live = pd.concat([
    live_arm("core_recall",    lambda s: LC.CorePolicy(s, lc_recall,  k_episodic=1)),
    live_arm("core_utility",   lambda s: LC.CorePolicy(s, lc_utility, k_episodic=1)),
    live_arm("forget_recall",  st=LC.subset_store(store, lk_recall)),
    live_arm("forget_utility", st=LC.subset_store(store, lk_utility)),
])
E.summarize(live, misranked)

In [ ]:
by_family(live)

In [ ]:
from pathlib import Path
OUT = Path("../data"); OUT.mkdir(exist_ok=True)
live.to_csv(OUT / "layer4_live_turns.csv", index=False)
live_scores.to_csv(OUT / "layer4_live_scores.csv", index=False)
print("saved")